In [10]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv(Path.cwd().parent / ".env")

url = (
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
engine = create_engine(url)

In [11]:
clientes = pd.read_sql("SELECT * FROM clientes", engine)
produtos = pd.read_sql("SELECT * FROM produtos", engine)
pedidos  = pd.read_sql("SELECT * FROM pedidos", engine)
itens    = pd.read_sql("SELECT * FROM itens_pedido", engine)

In [12]:
pedidos["data_pedido"] = pd.to_datetime(pedidos["data_pedido"])
clientes["data_cadastro"] = pd.to_datetime(clientes["data_cadastro"])

itens["valor_item"] = itens["quantidade"] * itens["preco_unitario"]
itens["valor_item"] = itens["valor_item"].astype(float)

In [13]:
base = (
    itens
    .merge(pedidos, on="id_pedido")
    .merge(produtos[["id_produto", "nome", "categoria"]], on="id_produto")
    .merge(clientes[["id_cliente", "nome", "estado"]], on="id_cliente",
           suffixes=("_produto", "_cliente"))
)

vendas = base[base["status"] != "cancelado"].copy()
print(base.shape, vendas.shape)

(7562, 13) (7040, 13)


In [14]:
sql_geral = """
SELECT
    SUM(i.quantidade * i.preco_unitario) AS faturamento,
    COUNT(DISTINCT p.id_pedido) AS pedidos
FROM itens_pedido i
JOIN pedidos p ON p.id_pedido = i.id_pedido
WHERE p.status <> 'cancelado'
"""
r = pd.read_sql(sql_geral, engine)
fat_sql = float(r.loc[0, "faturamento"])
ped_sql = int(r.loc[0, "pedidos"])

fat_pd = vendas["valor_item"].sum()
ped_pd = vendas["id_pedido"].nunique()

print("Faturamento  SQL:", round(fat_sql, 2), "| Pandas:", round(fat_pd, 2))
print("Pedidos      SQL:", ped_sql, "| Pandas:", ped_pd)
print("Ticket médio SQL:", round(fat_sql / ped_sql, 2),
      "| Pandas:", round(fat_pd / ped_pd, 2))

assert abs(fat_sql - fat_pd) < 0.01
assert ped_sql == ped_pd
print("✅ Validação 1 OK")

Faturamento  SQL: 1836430.2 | Pandas: 1836430.2
Pedidos      SQL: 2794 | Pandas: 2794
Ticket médio SQL: 657.28 | Pandas: 657.28
✅ Validação 1 OK


In [15]:
sql_cat = """
SELECT pr.categoria, SUM(i.quantidade * i.preco_unitario) AS faturamento
FROM itens_pedido i
JOIN pedidos p   ON p.id_pedido = i.id_pedido
JOIN produtos pr ON pr.id_produto = i.id_produto
WHERE p.status <> 'cancelado'
GROUP BY pr.categoria
"""
cat_sql = pd.read_sql(sql_cat, engine).set_index("categoria")["faturamento"].astype(float)
cat_pd = vendas.groupby("categoria")["valor_item"].sum()

comp = pd.concat([cat_sql, cat_pd], axis=1, keys=["sql", "pandas"])
comp["diferenca"] = (comp["sql"] - comp["pandas"]).abs()
print(comp.round(2))

assert comp["diferenca"].max() < 0.01
print("✅ Validação 2 OK")

                  sql    pandas  diferenca
categoria                                 
Moda         342466.1  342466.1        0.0
Casa         406784.1  406784.1        0.0
Esportes     436502.2  436502.2        0.0
Livros       164319.4  164319.4        0.0
Eletrônicos  486358.4  486358.4        0.0
✅ Validação 2 OK


In [16]:
sql_mes = """
SELECT DATE_TRUNC('month', p.data_pedido)::date AS mes,
       SUM(i.quantidade * i.preco_unitario) AS faturamento
FROM itens_pedido i
JOIN pedidos p ON p.id_pedido = i.id_pedido
WHERE p.status <> 'cancelado'
GROUP BY 1 ORDER BY 1
"""
mes_sql = pd.read_sql(sql_mes, engine)
mes_sql["mes"] = pd.to_datetime(mes_sql["mes"])
mes_sql = mes_sql.set_index("mes")["faturamento"].astype(float)

mes_pd = vendas.groupby(vendas["data_pedido"].dt.to_period("M"))["valor_item"].sum()
mes_pd.index = mes_pd.index.to_timestamp()

comp_mes = pd.concat([mes_sql, mes_pd], axis=1, keys=["sql", "pandas"])
comp_mes["diferenca"] = (comp_mes["sql"] - comp_mes["pandas"]).abs()
print(comp_mes.round(2))

assert len(mes_sql) == len(mes_pd)
assert comp_mes["diferenca"].max() < 0.01
print("✅ Validação 3 OK")

                 sql    pandas  diferenca
2024-01-01   63488.9   63488.9        0.0
2024-02-01   76159.4   76159.4        0.0
2024-03-01   91397.4   91397.4        0.0
2024-04-01   90642.6   90642.6        0.0
2024-05-01   95123.4   95123.4        0.0
2024-06-01   77929.5   77929.5        0.0
2024-07-01   79889.9   79889.9        0.0
2024-08-01   58588.7   58588.7        0.0
2024-09-01   75317.4   75317.4        0.0
2024-10-01   81725.2   81725.2        0.0
2024-11-01   82307.3   82307.3        0.0
2024-12-01   88567.3   88567.3        0.0
2025-01-01   96896.9   96896.9        0.0
2025-02-01   85863.7   85863.7        0.0
2025-03-01   80288.0   80288.0        0.0
2025-04-01   82293.0   82293.0        0.0
2025-05-01   92610.2   92610.2        0.0
2025-06-01   87283.2   87283.2        0.0
2025-07-01   86926.6   86926.6        0.0
2025-08-01  109171.3  109171.3        0.0
2025-09-01   79894.0   79894.0        0.0
2025-10-01   74066.3   74066.3        0.0
✅ Validação 3 OK


In [17]:
sql_prod = """
SELECT pr.nome, SUM(i.quantidade) AS qtd
FROM itens_pedido i
JOIN pedidos p   ON p.id_pedido = i.id_pedido
JOIN produtos pr ON pr.id_produto = i.id_produto
WHERE p.status <> 'cancelado'
GROUP BY pr.nome
ORDER BY qtd DESC, pr.nome
LIMIT 10
"""
prod_sql = pd.read_sql(sql_prod, engine).set_index("nome")["qtd"].astype(int)
prod_pd = (vendas.groupby("nome_produto")["quantidade"].sum()
           .sort_values(ascending=False).head(10))

print(pd.concat([prod_sql, prod_pd], axis=1, keys=["sql", "pandas"]))
assert prod_sql.sum() == prod_pd.sum()
print("✅ Validação 4 OK")

                    sql  pandas
Tapete de Yoga      417     417
Jogo de Panelas     398     398
Mochila             386     386
Aspirador Portátil  385     385
Bola de Futebol     379     379
Guia de Dados       377     377
Luminária LED       377     377
Carregador Turbo    371     371
Organizador         371     371
HQ Clássica         370     370
✅ Validação 4 OK


In [18]:
sql_cli = """
WITH receita_pedido AS (
    SELECT p.id_pedido, p.id_cliente,
           SUM(i.quantidade * i.preco_unitario) AS valor
    FROM pedidos p
    JOIN itens_pedido i ON i.id_pedido = p.id_pedido
    WHERE p.status <> 'cancelado'
    GROUP BY p.id_pedido, p.id_cliente
)
SELECT id_cliente, SUM(valor) AS total_gasto
FROM receita_pedido
GROUP BY id_cliente
ORDER BY total_gasto DESC
LIMIT 10
"""
cli_sql = pd.read_sql(sql_cli, engine)
cli_sql["total_gasto"] = cli_sql["total_gasto"].astype(float)

cli_pd = (vendas.groupby("id_cliente")["valor_item"].sum()
          .sort_values(ascending=False).head(10).reset_index())

comp_cli = cli_sql.merge(cli_pd, on="id_cliente", how="outer")
comp_cli = comp_cli.rename(columns={"total_gasto": "sql", "valor_item": "pandas"})
print(comp_cli.round(2))

assert abs(comp_cli["sql"].sum() - comp_cli["pandas"].sum()) < 0.01
print("✅ Validação 5 OK")

   id_cliente      sql   pandas
0          21   8745.1   8745.1
1          68   9616.1   9616.1
2          86  11068.8  11068.8
3         113   8510.0   8510.0
4         189   9888.0   9888.0
5         199   9119.2   9119.2
6         262   9232.0   9232.0
7         329   8566.0   8566.0
8         344  10903.3  10903.3
9         388  10655.5  10655.5
✅ Validação 5 OK
